# LAB 5 — Bài 2: Transformer cho bài toán Named Entity Recognition

> **Phiên bản hướng dẫn dành cho sinh viên mới học**  
> Notebook xây dựng một **Transformer Encoder từ đầu bằng PyTorch** và sử dụng mô hình để gán nhãn BIO cho từng token trong bộ dữ liệu PhoNER.

## Mục tiêu học tập

Sau khi hoàn thành bài này, sinh viên có thể:

1. Phân biệt **phân loại câu** và **gán nhãn chuỗi**.
2. Giải thích vai trò của embedding, positional encoding và multi-head self-attention.
3. Xây dựng vocabulary mà không làm rò rỉ dữ liệu từ dev/test.
4. Tạo batch có padding, attention mask và nhãn bỏ qua bằng `ignore_index`.
5. Huấn luyện Transformer cho NER bằng token-level cross entropy.
6. Đánh giá đúng bằng **entity-level precision, recall và F1**.
7. Phân tích lỗi BIO, token ngoài vocabulary và các câu dự đoán sai.

## Quy trình thực hành

```text
JSON Lines
   ↓
Kiểm tra words/tags và nhãn BIO
   ↓
Vocabulary + bảng nhãn từ train set
   ↓
Dataset → padding → attention mask
   ↓
Embedding + Positional Encoding
   ↓
Transformer Encoder
   ↓
Linear classifier cho từng token
   ↓
Entity-level F1 + phân tích lỗi
```


## 1. Cài đặt thư viện

Notebook sử dụng:

- `torch`: xây dựng và huấn luyện Transformer.
- `scikit-learn`: tạo confusion matrix ở mức token.
- `matplotlib`: trực quan hóa dữ liệu và quá trình học.

Các hàm entity-level precision, recall và F1 được cài đặt trực tiếp trong notebook. Nhờ đó, sinh viên có thể thấy rõ cách so khớp thực thể theo **loại + vị trí bắt đầu + vị trí kết thúc**, đồng thời notebook không phụ thuộc vào một thư viện đánh giá NER bên ngoài.

Chỉ cần chạy ô dưới đây khi môi trường chưa có các thư viện tương ứng.


In [ ]:
%pip install -q scikit-learn matplotlib


## 2. Import, thiết lập seed và chọn thiết bị

### Vì sao cần random seed?

Khởi tạo trọng số, xáo trộn dữ liệu và Dropout đều có yếu tố ngẫu nhiên. Đặt seed giúp kết quả giữa các lần chạy gần giống nhau hơn, thuận tiện khi kiểm tra bài thực hành.

> Seed làm tăng tính tái lập nhưng không bảo đảm mọi phần cứng và phiên bản thư viện cho kết quả giống tuyệt đối.


In [ ]:
from collections import Counter
from pathlib import Path
from typing import Any
import json
import math
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

SEED = 42


def set_seed(seed: int = SEED) -> None:
    """Thiết lập seed cho Python, NumPy và PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Hai thiết lập dưới đây ưu tiên tính tái lập hơn tốc độ.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FAST_DEV_RUN = os.getenv("FAST_DEV_RUN", "0") == "1"

print(f"Thiết bị: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"FAST_DEV_RUN: {FAST_DEV_RUN}")


## 3. Khai báo và kiểm tra đường dẫn dữ liệu

Notebook giả sử thư mục có cấu trúc:

```text
LAB5_Bai2_Transformer_NER.ipynb
PhoNER_COVID19/
├── train_word.json
├── dev_word.json
└── test_word.json
```

Mỗi tệp là **JSON Lines**; mỗi dòng là một đối tượng JSON độc lập.

Ta sử dụng:

- **Train set**: cập nhật trọng số, xây vocabulary và bảng nhãn.
- **Dev set**: chọn checkpoint, điều chỉnh siêu tham số và early stopping.
- **Test set**: đánh giá cuối cùng sau khi đã chọn mô hình.

Có thể đặt biến môi trường `PHONER_DIR` nếu dữ liệu nằm ở vị trí khác.


In [ ]:
DATA_DIR = Path(os.getenv("PHONER_DIR", "PhoNER_COVID19"))
TRAIN_PATH = DATA_DIR / "train_word.json"
DEV_PATH = DATA_DIR / "dev_word.json"
TEST_PATH = DATA_DIR / "test_word.json"

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "best_transformer_ner.pt"

required_paths = [TRAIN_PATH, DEV_PATH, TEST_PATH]
missing_paths = [path for path in required_paths if not path.exists()]

if missing_paths:
    missing_text = "\n".join(f"- {path}" for path in missing_paths)
    raise FileNotFoundError(
        "Không tìm thấy các tệp dữ liệu sau:\n"
        f"{missing_text}\n"
        "Hãy giải nén PhoNER_COVID19 đúng cấu trúc hoặc thiết lập PHONER_DIR."
    )

print("Đã tìm thấy đầy đủ dữ liệu:")
for path in required_paths:
    print(f"- {path}")


## 4. Đọc và kiểm tra dữ liệu JSON Lines

Mỗi mẫu được kỳ vọng có hai trường:

- `words`: danh sách token.
- `tags`: danh sách nhãn BIO tương ứng.

Hai danh sách phải có cùng độ dài. Một mẫu lệch token–nhãn sẽ làm sai toàn bộ bài toán gán nhãn chuỗi, vì vậy notebook kiểm tra ngay khi đọc.


In [ ]:
def read_jsonlines(path: Path) -> list[dict[str, list[str]]]:
    """Đọc JSON Lines và kiểm tra cấu trúc của từng mẫu."""
    records: list[dict[str, list[str]]] = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                sample = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"JSON không hợp lệ tại {path}, dòng {line_number}: {error}"
                ) from error

            words = sample.get("words")
            tags = sample.get("tags")

            if not isinstance(words, list) or not isinstance(tags, list):
                raise ValueError(
                    f"Mẫu tại {path}, dòng {line_number} phải chứa "
                    "hai danh sách 'words' và 'tags'."
                )

            if len(words) != len(tags):
                raise ValueError(
                    f"Lệch token–nhãn tại {path}, dòng {line_number}: "
                    f"{len(words)} words nhưng có {len(tags)} tags."
                )

            if not words:
                continue

            if not all(isinstance(item, str) for item in words + tags):
                raise ValueError(
                    f"Token và nhãn tại {path}, dòng {line_number} phải là chuỗi."
                )

            records.append({"words": words, "tags": tags})

    if not records:
        raise ValueError(f"Tệp {path} không có mẫu hợp lệ.")

    return records


train_records = read_jsonlines(TRAIN_PATH)
dev_records = read_jsonlines(DEV_PATH)
test_records = read_jsonlines(TEST_PATH)

print(f"Train: {len(train_records):,} câu")
print(f"Dev  : {len(dev_records):,} câu")
print(f"Test : {len(test_records):,} câu")

print("\nVí dụ đầu tiên:")
print(list(zip(train_records[0]["words"], train_records[0]["tags"])))


## 5. Khảo sát dữ liệu và kiểm tra nhãn BIO

Quy ước BIO:

- `B-X`: token bắt đầu một thực thể loại `X`.
- `I-X`: token tiếp tục thực thể loại `X`.
- `O`: token không thuộc thực thể.

Một lỗi BIO thường gặp là `I-X` xuất hiện sau `O` hoặc sau một thực thể khác loại. Ta kiểm tra sơ bộ trước khi huấn luyện.


In [ ]:
def summarize_split(name: str, records: list[dict[str, list[str]]]) -> dict[str, Any]:
    lengths = np.array([len(item["words"]) for item in records])

    return {
        "split": name,
        "sentences": len(records),
        "tokens": int(lengths.sum()),
        "average_length": float(lengths.mean()),
        "median_length": float(np.median(lengths)),
        "max_length": int(lengths.max()),
    }


def find_bio_issues(
    records: list[dict[str, list[str]]],
    maximum_examples: int = 10,
) -> list[tuple[int, int, str, str]]:
    """Tìm một số vị trí I-X không nối tiếp B-X/I-X cùng loại."""
    issues: list[tuple[int, int, str, str]] = []

    for sentence_index, sample in enumerate(records):
        previous_tag = "O"

        for token_index, current_tag in enumerate(sample["tags"]):
            if current_tag.startswith("I-"):
                entity_type = current_tag[2:]
                valid_previous = {
                    f"B-{entity_type}",
                    f"I-{entity_type}",
                }

                if previous_tag not in valid_previous:
                    issues.append(
                        (
                            sentence_index,
                            token_index,
                            sample["words"][token_index],
                            current_tag,
                        )
                    )

                    if len(issues) >= maximum_examples:
                        return issues

            previous_tag = current_tag

    return issues


summaries = [
    summarize_split("train", train_records),
    summarize_split("dev", dev_records),
    summarize_split("test", test_records),
]

for item in summaries:
    print(
        f"{item['split']:>5}: {item['sentences']:,} câu | "
        f"{item['tokens']:,} token | "
        f"trung bình {item['average_length']:.2f} | "
        f"dài nhất {item['max_length']}"
    )

tag_counter = Counter(
    tag
    for sample in train_records
    for tag in sample["tags"]
)

print("\nPhân bố nhãn trong train set:")
for tag, count in tag_counter.most_common():
    print(f"{tag:>24}: {count:,}")

bio_issues = find_bio_issues(train_records)
print(f"\nSố lỗi BIO minh họa tìm được: {len(bio_issues)}")
if bio_issues:
    print("Một số lỗi (sentence_index, token_index, token, tag):")
    for issue in bio_issues:
        print(issue)

lengths = [len(item["words"]) for item in train_records]
plt.figure(figsize=(8, 4))
plt.hist(lengths, bins=min(30, max(5, len(set(lengths)))))
plt.xlabel("Số token trong câu")
plt.ylabel("Số câu")
plt.title("Phân bố độ dài câu trong train set")
plt.show()


## 6. Xây dựng vocabulary và bảng nhãn

### Nguyên tắc quan trọng

Vocabulary và bảng nhãn chỉ được xây từ **train set**. Đọc từ hoặc nhãn của dev/test trước khi đánh giá là một dạng **data leakage**.

Hai token đặc biệt:

- `<pad>`: bù độ dài trong batch.
- `<unk>`: thay token chưa xuất hiện trong train set.

Nhãn padding không phải nhãn NER thật. Notebook dùng `LABEL_PAD_ID = -100` để `CrossEntropyLoss` bỏ qua các vị trí này.

### Hoa–thường

Tên riêng thường chứa thông tin viết hoa hữu ích, vì vậy mặc định giữ nguyên hoa–thường. Có thể đổi `LOWERCASE = True` để thực nghiệm.


In [ ]:
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_ID = 0
UNK_ID = 1
LABEL_PAD_ID = -100

LOWERCASE = False
MIN_FREQUENCY = 1


def normalize_token(token: str) -> str:
    return token.lower() if LOWERCASE else token


def build_vocabulary(
    records: list[dict[str, list[str]]],
    min_frequency: int = 1,
) -> tuple[dict[str, int], list[str]]:
    counter = Counter(
        normalize_token(token)
        for sample in records
        for token in sample["words"]
    )

    # Sắp xếp cố định để ánh xạ có tính tái lập.
    tokens = sorted(
        (
            token
            for token, frequency in counter.items()
            if frequency >= min_frequency
        ),
        key=lambda token: (-counter[token], token),
    )

    id_to_token = [PAD_TOKEN, UNK_TOKEN] + tokens
    token_to_id = {
        token: index
        for index, token in enumerate(id_to_token)
    }

    return token_to_id, id_to_token


def tag_sort_key(tag: str) -> tuple:
    if tag == "O":
        return ("", 0)

    if "-" not in tag:
        return (tag, 0)

    prefix, entity_type = tag.split("-", maxsplit=1)
    prefix_order = {"B": 0, "I": 1}.get(prefix, 2)
    return (entity_type, prefix_order)


def build_tag_mapping(
    records: list[dict[str, list[str]]],
) -> tuple[dict[str, int], list[str]]:
    tags = sorted(
        {
            tag
            for sample in records
            for tag in sample["tags"]
        },
        key=tag_sort_key,
    )

    id_to_tag = tags
    tag_to_id = {
        tag: index
        for index, tag in enumerate(id_to_tag)
    }

    return tag_to_id, id_to_tag


token_to_id, id_to_token = build_vocabulary(
    train_records,
    min_frequency=MIN_FREQUENCY,
)
tag_to_id, id_to_tag = build_tag_mapping(train_records)

train_tag_set = set(tag_to_id)
for split_name, records in [("dev", dev_records), ("test", test_records)]:
    unseen_tags = {
        tag
        for sample in records
        for tag in sample["tags"]
        if tag not in train_tag_set
    }

    if unseen_tags:
        raise ValueError(
            f"{split_name} có nhãn chưa xuất hiện trong train set: "
            f"{sorted(unseen_tags)}"
        )

print(f"Kích thước vocabulary: {len(token_to_id):,}")
print(f"Số nhãn NER: {len(tag_to_id)}")
print(f"Các nhãn: {id_to_tag}")


## 7. Tạo `Dataset` và hàm `collate_fn`

Các câu có độ dài khác nhau nên không thể ghép trực tiếp thành tensor hình chữ nhật.

`collate_fn` thực hiện:

1. Padding `input_ids` bằng `PAD_ID`.
2. Padding `labels` bằng `LABEL_PAD_ID`.
3. Tạo `attention_mask`: `True` tại token thật, `False` tại padding.
4. Giữ lại token và nhãn dạng chuỗi để phân tích lỗi.

`MAX_LENGTH` giới hạn độ dài để positional encoding và bộ nhớ GPU luôn kiểm soát được.


In [ ]:
MAX_LENGTH = 256


class NERDataset(Dataset):
    def __init__(
        self,
        records: list[dict[str, list[str]]],
        token_to_id: dict[str, int],
        tag_to_id: dict[str, int],
        max_length: int,
    ) -> None:
        self.records = records
        self.token_to_id = token_to_id
        self.tag_to_id = tag_to_id
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> dict[str, Any]:
        sample = self.records[index]
        words = sample["words"][: self.max_length]
        tags = sample["tags"][: self.max_length]

        input_ids = [
            self.token_to_id.get(normalize_token(token), UNK_ID)
            for token in words
        ]
        label_ids = [
            self.tag_to_id[tag]
            for tag in tags
        ]

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(label_ids, dtype=torch.long),
            "tokens": words,
            "tags": tags,
        }


def collate_ner_batch(batch: list[dict[str, Any]]) -> dict[str, Any]:
    input_ids = pad_sequence(
        [item["input_ids"] for item in batch],
        batch_first=True,
        padding_value=PAD_ID,
    )
    labels = pad_sequence(
        [item["labels"] for item in batch],
        batch_first=True,
        padding_value=LABEL_PAD_ID,
    )
    attention_mask = input_ids.ne(PAD_ID)
    lengths = attention_mask.sum(dim=1)

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask,
        "lengths": lengths,
        "tokens": [item["tokens"] for item in batch],
        "tags": [item["tags"] for item in batch],
    }


train_dataset = NERDataset(
    train_records,
    token_to_id,
    tag_to_id,
    MAX_LENGTH,
)
dev_dataset = NERDataset(
    dev_records,
    token_to_id,
    tag_to_id,
    MAX_LENGTH,
)
test_dataset = NERDataset(
    test_records,
    token_to_id,
    tag_to_id,
    MAX_LENGTH,
)

sample = train_dataset[0]
print("input_ids:", sample["input_ids"])
print("labels   :", sample["labels"])
print("tokens   :", sample["tokens"])
print("tags     :", sample["tags"])


## 8. Tạo `DataLoader` và kiểm tra một batch

- Train loader bật `shuffle=True`.
- Dev/test giữ nguyên thứ tự để phân tích kết quả dễ hơn.
- `pin_memory=True` chỉ hữu ích khi dùng CUDA.
- `num_workers=0` tương thích tốt với notebook trên nhiều hệ điều hành.


In [ ]:
BATCH_SIZE = 4 if FAST_DEV_RUN else 32

loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": 0,
    "pin_memory": DEVICE.type == "cuda",
    "collate_fn": collate_ner_batch,
}

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    **loader_kwargs,
)
dev_loader = DataLoader(
    dev_dataset,
    shuffle=False,
    **loader_kwargs,
)
test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    **loader_kwargs,
)

batch = next(iter(train_loader))

print("input_ids     :", tuple(batch["input_ids"].shape))
print("labels        :", tuple(batch["labels"].shape))
print("attention_mask:", tuple(batch["attention_mask"].shape))
print("lengths       :", batch["lengths"].tolist())

assert batch["input_ids"].shape == batch["labels"].shape
assert batch["attention_mask"].dtype == torch.bool
assert torch.equal(
    batch["attention_mask"],
    batch["input_ids"].ne(PAD_ID),
)


## 9. Positional Encoding

Self-attention xử lý tất cả token song song nên bản thân phép toán không biết thứ tự token.

Positional encoding hình sin–cos được cộng vào embedding:

\[
PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d}}\right)
\]

\[
PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d}}\right)
\]

Notebook còn:

- Nhân embedding với \(\sqrt{d_{model}}\), như Transformer gốc.
- Dùng dropout sau khi cộng vị trí.
- Đưa vector padding về 0 để các vị trí giả không tiếp tục lan truyền tín hiệu.


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(
        self,
        d_model: int,
        dropout: float,
        max_length: int,
    ) -> None:
        super().__init__()

        position = torch.arange(
            max_length,
            dtype=torch.float,
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / d_model)
        )

        encoding = torch.zeros(max_length, d_model)
        encoding[:, 0::2] = torch.sin(position * div_term)

        # Dòng này vẫn hoạt động khi d_model là số lẻ.
        odd_width = encoding[:, 1::2].shape[1]
        encoding[:, 1::2] = torch.cos(
            position * div_term[:odd_width]
        )

        self.register_buffer(
            "encoding",
            encoding.unsqueeze(0),
            persistent=False,
        )
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        sequence_length = x.size(1)

        if sequence_length > self.encoding.size(1):
            raise ValueError(
                f"Chuỗi dài {sequence_length}, vượt max_length="
                f"{self.encoding.size(1)}."
            )

        x = x + self.encoding[:, :sequence_length]
        x = self.dropout(x)
        return x * attention_mask.unsqueeze(-1)


## 10. Multi-Head Self-Attention

Từ cùng đầu vào \(X\), mô hình tạo Query, Key và Value:

\[
Q=XW_Q,\quad K=XW_K,\quad V=XW_V
\]

Scaled dot-product attention:

\[
Attention(Q,K,V)
=
softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

Multi-head attention chia không gian biểu diễn thành nhiều head để quan sát nhiều kiểu quan hệ token khác nhau.

### Hai loại mask cần phân biệt

- **Key padding mask**: không cho các token thật chú ý đến `<pad>`.
- **Query mask**: đưa đầu ra tại vị trí `<pad>` về 0.

NER dùng toàn bộ câu ở cả hai phía nên **không dùng causal mask**.


In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        dropout: float,
    ) -> None:
        super().__init__()

        if d_model % num_heads != 0:
            raise ValueError(
                "d_model phải chia hết cho num_heads."
            )

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = math.sqrt(self.head_dim)

        self.qkv_projection = nn.Linear(
            d_model,
            3 * d_model,
        )
        self.output_projection = nn.Linear(
            d_model,
            d_model,
        )
        self.attention_dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        attention_mask: torch.Tensor,
        return_attention: bool = False,
    ) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:
        batch_size, sequence_length, _ = x.shape

        qkv = self.qkv_projection(x)
        qkv = qkv.view(
            batch_size,
            sequence_length,
            3,
            self.num_heads,
            self.head_dim,
        )
        qkv = qkv.permute(2, 0, 3, 1, 4)
        query, key, value = qkv.unbind(dim=0)

        scores = torch.matmul(
            query,
            key.transpose(-2, -1),
        ) / self.scale

        key_mask = attention_mask[:, None, None, :]
        scores = scores.masked_fill(
            ~key_mask,
            torch.finfo(scores.dtype).min,
        )

        attention_weights = torch.softmax(
            scores,
            dim=-1,
        )
        attention_weights = self.attention_dropout(
            attention_weights
        )

        context = torch.matmul(
            attention_weights,
            value,
        )
        context = context.transpose(1, 2).contiguous()
        context = context.view(
            batch_size,
            sequence_length,
            self.d_model,
        )

        output = self.output_projection(context)
        output = output * attention_mask.unsqueeze(-1)

        if return_attention:
            return output, attention_weights

        return output


## 11. Transformer Encoder Layer

Mỗi encoder layer gồm:

1. Multi-head self-attention.
2. Residual connection.
3. Layer normalization.
4. Feed-forward network áp dụng độc lập tại từng token.
5. Dropout.

Notebook dùng **Pre-LayerNorm**: chuẩn hóa trước attention và feed-forward. Cách này thường ổn định hơn khi huấn luyện mô hình từ đầu.

Feed-forward sử dụng `GELU`, gần với các Transformer hiện đại hơn so với `ReLU`.


In [ ]:
class TransformerEncoderLayer(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        dropout: float,
    ) -> None:
        super().__init__()

        self.norm1 = nn.LayerNorm(d_model)
        self.self_attention = MultiHeadSelfAttention(
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout,
        )
        self.norm2 = nn.LayerNorm(d_model)

        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

        self.residual_dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        attention_output = self.self_attention(
            self.norm1(x),
            attention_mask,
        )
        x = x + self.residual_dropout(attention_output)

        feed_forward_output = self.feed_forward(
            self.norm2(x)
        )
        x = x + self.residual_dropout(
            feed_forward_output
        )

        return x * attention_mask.unsqueeze(-1)


class TransformerEncoder(nn.Module):
    def __init__(
        self,
        num_layers: int,
        d_model: int,
        num_heads: int,
        d_ff: int,
        dropout: float,
    ) -> None:
        super().__init__()

        self.layers = nn.ModuleList(
            [
                TransformerEncoderLayer(
                    d_model=d_model,
                    num_heads=num_heads,
                    d_ff=d_ff,
                    dropout=dropout,
                )
                for _ in range(num_layers)
            ]
        )
        self.final_norm = nn.LayerNorm(d_model)

    def forward(
        self,
        x: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, attention_mask)

        x = self.final_norm(x)
        return x * attention_mask.unsqueeze(-1)


## 12. Mô hình Transformer cho NER

Luồng dữ liệu:

```text
input_ids
   ↓
Token Embedding × √d_model
   ↓
Positional Encoding
   ↓
N Transformer Encoder Layers
   ↓
Linear(d_model → số nhãn)
   ↓
logits cho từng token
```

Mỗi token được phân loại độc lập ở tầng cuối. Self-attention giúp biểu diễn của token đã chứa ngữ cảnh toàn câu.

> Khác BiLSTM-CRF, mô hình này chưa có tầng CRF nên vẫn có thể sinh chuỗi BIO không hợp lệ. Ta sẽ đo lỗi này ở phần phân tích.


In [ ]:
class TransformerNER(nn.Module):
    def __init__(
        self,
        vocabulary_size: int,
        number_of_tags: int,
        d_model: int,
        num_heads: int,
        num_layers: int,
        d_ff: int,
        dropout: float,
        max_length: int,
    ) -> None:
        super().__init__()

        self.d_model = d_model
        self.embedding = nn.Embedding(
            vocabulary_size,
            d_model,
            padding_idx=PAD_ID,
        )
        self.positional_encoding = PositionalEncoding(
            d_model=d_model,
            dropout=dropout,
            max_length=max_length,
        )
        self.encoder = TransformerEncoder(
            num_layers=num_layers,
            d_model=d_model,
            num_heads=num_heads,
            d_ff=d_ff,
            dropout=dropout,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, number_of_tags),
        )

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        if attention_mask is None:
            attention_mask = input_ids.ne(PAD_ID)

        x = self.embedding(input_ids)
        x = x * math.sqrt(self.d_model)
        x = self.positional_encoding(
            x,
            attention_mask,
        )
        x = self.encoder(
            x,
            attention_mask,
        )
        logits = self.classifier(x)

        return logits


## 13. Khởi tạo mô hình và kiểm tra forward pass

Cấu hình mặc định đủ nhỏ cho bài thực hành nhưng vẫn minh họa đầy đủ Transformer Encoder.

Khi `FAST_DEV_RUN=1`, notebook tự giảm kích thước mô hình để kiểm thử nhanh.


In [ ]:
MODEL_CONFIG = {
    "d_model": 32 if FAST_DEV_RUN else 128,
    "num_heads": 4,
    "num_layers": 1 if FAST_DEV_RUN else 3,
    "d_ff": 64 if FAST_DEV_RUN else 256,
    "dropout": 0.1,
    "max_length": MAX_LENGTH,
}

model = TransformerNER(
    vocabulary_size=len(token_to_id),
    number_of_tags=len(tag_to_id),
    **MODEL_CONFIG,
).to(DEVICE)


def count_trainable_parameters(model: nn.Module) -> int:
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


batch_on_device = {
    key: value.to(DEVICE)
    for key, value in batch.items()
    if isinstance(value, torch.Tensor)
}

with torch.no_grad():
    logits = model(
        batch_on_device["input_ids"],
        batch_on_device["attention_mask"],
    )

print(model)
print(f"\nSố tham số có thể học: {count_trainable_parameters(model):,}")
print(f"Logits shape: {tuple(logits.shape)}")

expected_shape = (
    batch["input_ids"].size(0),
    batch["input_ids"].size(1),
    len(tag_to_id),
)
assert logits.shape == expected_shape


## 14. Loss, optimizer và learning-rate scheduler

### Loss

Dùng `CrossEntropyLoss(ignore_index=-100)` để bỏ qua nhãn padding.

### Optimizer

`AdamW` tách weight decay khỏi bước cập nhật Adam, phù hợp với Transformer.

### Scheduler

`ReduceLROnPlateau` giảm learning rate khi entity-level F1 trên dev set không tăng.

### Lưu ý về mất cân bằng nhãn

NER thường có rất nhiều nhãn `O`. Notebook không dùng class weight mặc định để giữ bài học đơn giản; sinh viên có thể thử ở phần bài tập mở rộng.


In [ ]:
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-2
GRADIENT_CLIP = 1.0
EPOCHS = 2 if FAST_DEV_RUN else 25
PATIENCE = 2 if FAST_DEV_RUN else 5

loss_function = nn.CrossEntropyLoss(
    ignore_index=LABEL_PAD_ID,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)


## 15. Hàm huấn luyện một epoch

Một lỗi phổ biến là nhân loss trung bình với `y.numel()`, trong đó có cả padding. Cách đúng là đếm số token có nhãn khác `LABEL_PAD_ID`.

Quy trình cho mỗi batch:

1. Chuyển tensor sang thiết bị.
2. Xóa gradient cũ.
3. Forward pass.
4. Tính token-level loss.
5. Backpropagation.
6. Gradient clipping.
7. Cập nhật trọng số.


In [ ]:
def move_batch_to_device(
    batch: dict[str, Any],
    device: torch.device,
) -> dict[str, Any]:
    return {
        key: value.to(
            device,
            non_blocking=device.type == "cuda",
        )
        if isinstance(value, torch.Tensor)
        else value
        for key, value in batch.items()
    }


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_function: nn.Module,
    device: torch.device,
    gradient_clip: float,
) -> float:
    model.train()

    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        batch = move_batch_to_device(batch, device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(
            batch["input_ids"],
            batch["attention_mask"],
        )
        loss = loss_function(
            logits.reshape(-1, logits.size(-1)),
            batch["labels"].reshape(-1),
        )

        loss.backward()
        nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=gradient_clip,
        )
        optimizer.step()

        number_of_tokens = (
            batch["labels"] != LABEL_PAD_ID
        ).sum().item()

        total_loss += loss.item() * number_of_tokens
        total_tokens += number_of_tokens

    return total_loss / max(total_tokens, 1)


## 16. Hàm đánh giá loss và entity-level metrics

Token accuracy có thể rất cao chỉ vì nhãn `O` chiếm đa số. Do đó, chỉ số chính là:

- **Precision**: trong các thực thể mô hình dự đoán, bao nhiêu thực thể đúng.
- **Recall**: trong các thực thể thật, mô hình tìm được bao nhiêu.
- **F1**: trung bình điều hòa giữa precision và recall.

Một thực thể chỉ được tính đúng khi đồng thời đúng:

1. Loại thực thể.
2. Vị trí token bắt đầu.
3. Vị trí token kết thúc.

Ta tự cài đặt bộ đánh giá để thấy rõ nguyên tắc này. Padding được cắt bằng `attention_mask` trước khi tính.


In [ ]:
def get_entities(
    tags: list[str],
) -> list[tuple[str, int, int]]:
    """Chuyển một chuỗi BIO thành các bộ (loại, bắt đầu, kết thúc)."""
    entities: list[tuple[str, int, int]] = []
    current_type: str | None = None
    start_index: int | None = None

    for index in range(len(tags) + 1):
        tag = tags[index] if index < len(tags) else "O"

        if tag == "O" or "-" not in tag:
            prefix = "O"
            entity_type = None
        else:
            prefix, entity_type = tag.split("-", maxsplit=1)

        should_close = (
            current_type is not None
            and (
                prefix in {"O", "B"}
                or entity_type != current_type
            )
        )

        if should_close:
            entities.append(
                (
                    current_type,
                    int(start_index),
                    index - 1,
                )
            )
            current_type = None
            start_index = None

        if prefix == "B":
            current_type = entity_type
            start_index = index
        elif prefix == "I":
            # I-X không nối tiếp X hợp lệ được xem như bắt đầu một chunk mới.
            if current_type is None:
                current_type = entity_type
                start_index = index
        elif prefix == "O":
            current_type = None
            start_index = None

    return entities


def build_entity_set(
    tag_sequences: list[list[str]],
) -> set[tuple[int, str, int, int]]:
    entity_set: set[tuple[int, str, int, int]] = set()

    for sentence_index, tags in enumerate(tag_sequences):
        for entity_type, start, end in get_entities(tags):
            entity_set.add(
                (
                    sentence_index,
                    entity_type,
                    start,
                    end,
                )
            )

    return entity_set


def calculate_entity_metrics(
    true_sequences: list[list[str]],
    predicted_sequences: list[list[str]],
) -> dict[str, float]:
    true_entities = build_entity_set(true_sequences)
    predicted_entities = build_entity_set(predicted_sequences)

    true_positive = len(true_entities & predicted_entities)
    false_positive = len(predicted_entities - true_entities)
    false_negative = len(true_entities - predicted_entities)

    precision = (
        true_positive / (true_positive + false_positive)
        if true_positive + false_positive > 0
        else 0.0
    )
    recall = (
        true_positive / (true_positive + false_negative)
        if true_positive + false_negative > 0
        else 0.0
    )
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative,
    }


def calculate_token_accuracy(
    true_sequences: list[list[str]],
    predicted_sequences: list[list[str]],
) -> float:
    correct = 0
    total = 0

    for true_tags, predicted_tags in zip(
        true_sequences,
        predicted_sequences,
    ):
        for true_tag, predicted_tag in zip(
            true_tags,
            predicted_tags,
        ):
            correct += int(true_tag == predicted_tag)
            total += 1

    return correct / total if total > 0 else 0.0


@torch.no_grad()
def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    loss_function: nn.Module,
    device: torch.device,
    id_to_tag: list[str],
) -> dict[str, Any]:
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    all_true_tags: list[list[str]] = []
    all_predicted_tags: list[list[str]] = []
    all_tokens: list[list[str]] = []

    for batch in loader:
        raw_tokens = batch["tokens"]
        batch = move_batch_to_device(batch, device)

        logits = model(
            batch["input_ids"],
            batch["attention_mask"],
        )
        loss = loss_function(
            logits.reshape(-1, logits.size(-1)),
            batch["labels"].reshape(-1),
        )

        predictions = logits.argmax(dim=-1)
        number_of_tokens = batch["attention_mask"].sum().item()

        total_loss += loss.item() * number_of_tokens
        total_tokens += number_of_tokens

        for row_index, length in enumerate(batch["lengths"].tolist()):
            true_ids = batch["labels"][
                row_index, :length
            ].tolist()
            predicted_ids = predictions[
                row_index, :length
            ].tolist()

            all_true_tags.append(
                [id_to_tag[tag_id] for tag_id in true_ids]
            )
            all_predicted_tags.append(
                [id_to_tag[tag_id] for tag_id in predicted_ids]
            )
            all_tokens.append(raw_tokens[row_index][:length])

    entity_metrics = calculate_entity_metrics(
        all_true_tags,
        all_predicted_tags,
    )

    metrics = {
        "loss": total_loss / max(total_tokens, 1),
        "accuracy": calculate_token_accuracy(
            all_true_tags,
            all_predicted_tags,
        ),
        **entity_metrics,
        "true_tags": all_true_tags,
        "predicted_tags": all_predicted_tags,
        "tokens": all_tokens,
    }

    return metrics


## 17. Vòng lặp huấn luyện, checkpoint và early stopping

Checkpoint được chọn theo **dev entity-level F1**, không theo test set.

Checkpoint lưu:

- Trọng số mô hình.
- Trạng thái optimizer và scheduler.
- Cấu hình mô hình.
- Vocabulary và bảng nhãn.
- Epoch cùng các chỉ số tốt nhất.

Early stopping dừng khi F1 không cải thiện trong `PATIENCE` epoch liên tiếp.


In [ ]:
history = {
    "train_loss": [],
    "dev_loss": [],
    "dev_f1": [],
    "dev_precision": [],
    "dev_recall": [],
    "learning_rate": [],
}

best_dev_f1 = -1.0
best_dev_loss = float("inf")
epochs_without_improvement = 0

training_start = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        loss_function=loss_function,
        device=DEVICE,
        gradient_clip=GRADIENT_CLIP,
    )

    dev_metrics = evaluate_model(
        model=model,
        loader=dev_loader,
        loss_function=loss_function,
        device=DEVICE,
        id_to_tag=id_to_tag,
    )

    scheduler.step(dev_metrics["f1"])
    current_learning_rate = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["dev_loss"].append(dev_metrics["loss"])
    history["dev_f1"].append(dev_metrics["f1"])
    history["dev_precision"].append(dev_metrics["precision"])
    history["dev_recall"].append(dev_metrics["recall"])
    history["learning_rate"].append(current_learning_rate)

    improved = (
        dev_metrics["f1"] > best_dev_f1 + 1e-8
        or (
            abs(dev_metrics["f1"] - best_dev_f1) <= 1e-8
            and dev_metrics["loss"] < best_dev_loss
        )
    )

    if improved:
        best_dev_f1 = dev_metrics["f1"]
        best_dev_loss = dev_metrics["loss"]
        epochs_without_improvement = 0

        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "model_config": MODEL_CONFIG,
            "token_to_id": token_to_id,
            "id_to_token": id_to_token,
            "tag_to_id": tag_to_id,
            "id_to_tag": id_to_tag,
            "lowercase": LOWERCASE,
            "max_length": MAX_LENGTH,
            "best_dev_f1": best_dev_f1,
            "best_dev_loss": best_dev_loss,
        }
        torch.save(checkpoint, CHECKPOINT_PATH)
    else:
        epochs_without_improvement += 1

    elapsed = time.time() - epoch_start

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {train_loss:.4f} | "
        f"dev loss {dev_metrics['loss']:.4f} | "
        f"P {dev_metrics['precision']:.4f} | "
        f"R {dev_metrics['recall']:.4f} | "
        f"F1 {dev_metrics['f1']:.4f} | "
        f"lr {current_learning_rate:.2e} | "
        f"{elapsed:.1f}s"
    )

    if epochs_without_improvement >= PATIENCE:
        print(
            f"Early stopping: dev F1 không cải thiện "
            f"trong {PATIENCE} epoch."
        )
        break

print(
    f"\nHuấn luyện hoàn tất sau "
    f"{time.time() - training_start:.1f} giây."
)
print(f"Checkpoint tốt nhất: {CHECKPOINT_PATH}")
print(f"Best dev F1: {best_dev_f1:.4f}")


## 18. Trực quan hóa quá trình học

Quan sát đồng thời:

- Train loss và dev loss.
- Dev precision, recall và F1.
- Learning rate.

Dấu hiệu overfitting thường gặp: train loss tiếp tục giảm nhưng dev loss tăng hoặc dev F1 giảm.


In [ ]:
epochs_ran = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(8, 4))
plt.plot(epochs_ran, history["train_loss"], marker="o", label="Train loss")
plt.plot(epochs_ran, history["dev_loss"], marker="o", label="Dev loss")
plt.xlabel("Epoch")
plt.ylabel("Loss trung bình trên token thật")
plt.title("Loss theo epoch")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(epochs_ran, history["dev_precision"], marker="o", label="Precision")
plt.plot(epochs_ran, history["dev_recall"], marker="o", label="Recall")
plt.plot(epochs_ran, history["dev_f1"], marker="o", label="Entity F1")
plt.xlabel("Epoch")
plt.ylabel("Điểm số")
plt.ylim(0, 1.05)
plt.title("Chỉ số trên dev set")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(epochs_ran, history["learning_rate"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Learning rate")
plt.title("Learning rate theo epoch")
plt.grid(alpha=0.3)
plt.show()


## 19. Tải checkpoint tốt nhất và đánh giá test set

Không nên dùng trọng số của epoch cuối cùng nếu checkpoint tốt nhất xuất hiện sớm hơn.

Ta tải lại checkpoint được chọn bằng dev set rồi mới đánh giá test set đúng một lần.


In [ ]:
def load_checkpoint(
    path: Path,
    device: torch.device,
) -> dict[str, Any]:
    try:
        return torch.load(
            path,
            map_location=device,
            weights_only=False,
        )
    except TypeError:
        # Tương thích với PyTorch cũ chưa có tham số weights_only.
        return torch.load(
            path,
            map_location=device,
        )


best_checkpoint = load_checkpoint(
    CHECKPOINT_PATH,
    DEVICE,
)

best_model = TransformerNER(
    vocabulary_size=len(best_checkpoint["token_to_id"]),
    number_of_tags=len(best_checkpoint["id_to_tag"]),
    **best_checkpoint["model_config"],
).to(DEVICE)

best_model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

test_metrics = evaluate_model(
    model=best_model,
    loader=test_loader,
    loss_function=loss_function,
    device=DEVICE,
    id_to_tag=best_checkpoint["id_to_tag"],
)

print(f"Checkpoint epoch: {best_checkpoint['epoch']}")
print(f"Test loss      : {test_metrics['loss']:.4f}")
print(f"Token accuracy : {test_metrics['accuracy']:.4f}")
print(f"Entity precision: {test_metrics['precision']:.4f}")
print(f"Entity recall   : {test_metrics['recall']:.4f}")
print(f"Entity F1       : {test_metrics['f1']:.4f}")


## 20. Classification report theo loại thực thể

Báo cáo dưới đây đánh giá **thực thể hoàn chỉnh**, nghĩa là cả loại và biên thực thể phải đúng.

Ví dụ, dự đoán đúng hai trong ba token của tên tổ chức vẫn chưa được tính là một thực thể đúng hoàn toàn.


In [ ]:
def entity_classification_report(
    true_sequences: list[list[str]],
    predicted_sequences: list[list[str]],
    digits: int = 4,
) -> str:
    true_entities = build_entity_set(true_sequences)
    predicted_entities = build_entity_set(predicted_sequences)

    entity_types = sorted(
        {
            entity[1]
            for entity in true_entities | predicted_entities
        }
    )

    header = (
        f"{'entity type':<24}"
        f"{'precision':>11}"
        f"{'recall':>11}"
        f"{'f1-score':>11}"
        f"{'support':>11}"
    )
    lines = [header, "-" * len(header)]

    total_true_positive = 0
    total_false_positive = 0
    total_false_negative = 0
    per_type_scores = []

    for entity_type in entity_types:
        true_of_type = {
            entity
            for entity in true_entities
            if entity[1] == entity_type
        }
        predicted_of_type = {
            entity
            for entity in predicted_entities
            if entity[1] == entity_type
        }

        true_positive = len(true_of_type & predicted_of_type)
        false_positive = len(predicted_of_type - true_of_type)
        false_negative = len(true_of_type - predicted_of_type)

        precision = (
            true_positive / (true_positive + false_positive)
            if true_positive + false_positive > 0
            else 0.0
        )
        recall = (
            true_positive / (true_positive + false_negative)
            if true_positive + false_negative > 0
            else 0.0
        )
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0
            else 0.0
        )
        support = len(true_of_type)

        total_true_positive += true_positive
        total_false_positive += false_positive
        total_false_negative += false_negative
        per_type_scores.append((precision, recall, f1, support))

        lines.append(
            f"{entity_type:<24}"
            f"{precision:>11.{digits}f}"
            f"{recall:>11.{digits}f}"
            f"{f1:>11.{digits}f}"
            f"{support:>11d}"
        )

    total_support = sum(item[3] for item in per_type_scores)

    macro_precision = (
        np.mean([item[0] for item in per_type_scores])
        if per_type_scores
        else 0.0
    )
    macro_recall = (
        np.mean([item[1] for item in per_type_scores])
        if per_type_scores
        else 0.0
    )
    macro_f1 = (
        np.mean([item[2] for item in per_type_scores])
        if per_type_scores
        else 0.0
    )

    micro_precision = (
        total_true_positive
        / (total_true_positive + total_false_positive)
        if total_true_positive + total_false_positive > 0
        else 0.0
    )
    micro_recall = (
        total_true_positive
        / (total_true_positive + total_false_negative)
        if total_true_positive + total_false_negative > 0
        else 0.0
    )
    micro_f1 = (
        2 * micro_precision * micro_recall
        / (micro_precision + micro_recall)
        if micro_precision + micro_recall > 0
        else 0.0
    )

    lines.extend(
        [
            "",
            f"{'micro avg':<24}"
            f"{micro_precision:>11.{digits}f}"
            f"{micro_recall:>11.{digits}f}"
            f"{micro_f1:>11.{digits}f}"
            f"{total_support:>11d}",
            f"{'macro avg':<24}"
            f"{macro_precision:>11.{digits}f}"
            f"{macro_recall:>11.{digits}f}"
            f"{macro_f1:>11.{digits}f}"
            f"{total_support:>11d}",
        ]
    )

    return "\n".join(lines)


print(
    entity_classification_report(
        test_metrics["true_tags"],
        test_metrics["predicted_tags"],
        digits=4,
    )
)


## 21. Confusion matrix ở mức token

Confusion matrix giúp thấy mô hình hay nhầm giữa các nhãn token nào, nhưng không thay thế entity-level F1.

Để biểu đồ dễ đọc, notebook chỉ dùng các nhãn có xuất hiện trong nhãn thật hoặc dự đoán.


In [ ]:
flat_true_tags = [
    tag
    for sentence in test_metrics["true_tags"]
    for tag in sentence
]
flat_predicted_tags = [
    tag
    for sentence in test_metrics["predicted_tags"]
    for tag in sentence
]

active_tags = [
    tag
    for tag in id_to_tag
    if tag in set(flat_true_tags) | set(flat_predicted_tags)
]

matrix = confusion_matrix(
    flat_true_tags,
    flat_predicted_tags,
    labels=active_tags,
)

figure_size = max(8, len(active_tags) * 0.75)
fig, ax = plt.subplots(
    figsize=(figure_size, figure_size)
)
display_object = ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=active_tags,
)
display_object.plot(
    ax=ax,
    xticks_rotation=90,
    values_format="d",
    colorbar=False,
)
ax.set_title("Confusion matrix ở mức token")
plt.tight_layout()
plt.show()


## 22. Phân tích lỗi và kiểm tra chuỗi BIO dự đoán

Ta xem các câu có nhiều token sai nhất và đo số lỗi BIO.

### Vì sao Transformer + Linear có thể vi phạm BIO?

Tầng linear dự đoán từng token độc lập. Mô hình có thể chọn `I-LOCATION` ngay sau `O` dù chuỗi này không hợp lệ về mặt cấu trúc.

Một hướng cải tiến là đặt **CRF** lên trên Transformer.


In [ ]:
def count_bio_violations(tag_sequences: list[list[str]]) -> int:
    violations = 0

    for tags in tag_sequences:
        previous_tag = "O"

        for current_tag in tags:
            if current_tag.startswith("I-"):
                entity_type = current_tag[2:]
                if previous_tag not in {
                    f"B-{entity_type}",
                    f"I-{entity_type}",
                }:
                    violations += 1

            previous_tag = current_tag

    return violations


def sentence_error_count(
    true_tags: list[str],
    predicted_tags: list[str],
) -> int:
    return sum(
        true_tag != predicted_tag
        for true_tag, predicted_tag in zip(
            true_tags,
            predicted_tags,
        )
    )


ranked_errors = sorted(
    range(len(test_metrics["tokens"])),
    key=lambda index: sentence_error_count(
        test_metrics["true_tags"][index],
        test_metrics["predicted_tags"][index],
    ),
    reverse=True,
)

print(
    "Số lỗi BIO trong dự đoán:",
    count_bio_violations(
        test_metrics["predicted_tags"]
    ),
)

maximum_examples = min(5, len(ranked_errors))

for rank, sentence_index in enumerate(
    ranked_errors[:maximum_examples],
    start=1,
):
    tokens = test_metrics["tokens"][sentence_index]
    true_tags = test_metrics["true_tags"][sentence_index]
    predicted_tags = test_metrics["predicted_tags"][sentence_index]

    mistakes = sentence_error_count(
        true_tags,
        predicted_tags,
    )

    print(
        f"\nVí dụ {rank} — "
        f"{mistakes}/{len(tokens)} token sai"
    )
    print(
        f"{'TOKEN':<24}"
        f"{'TRUE':<24}"
        f"{'PREDICTED':<24}"
        "OOV"
    )
    print("-" * 82)

    for token, true_tag, predicted_tag in zip(
        tokens,
        true_tags,
        predicted_tags,
    ):
        oov = (
            normalize_token(token)
            not in token_to_id
        )
        marker = "*" if true_tag != predicted_tag else ""

        print(
            f"{token:<24}"
            f"{true_tag:<24}"
            f"{predicted_tag:<24}"
            f"{str(oov):<5} {marker}"
        )


## 23. Dự đoán câu mới, tổng kết và bài tập mở rộng

Hàm dưới đây nhận một **danh sách token đã tách sẵn**. Đây là lựa chọn an toàn cho tiếng Việt vì cách tách từ phải phù hợp với dữ liệu huấn luyện.

Kết quả trả về:

- Nhãn dự đoán cho từng token.
- Danh sách thực thể được ghép từ chuỗi BIO.

### Tổng kết

Notebook đã hoàn thành toàn bộ pipeline:

1. Đọc và kiểm tra dữ liệu.
2. Xây vocabulary chỉ từ train set.
3. Tạo padding, label padding và attention mask.
4. Cài đặt Transformer Encoder từ đầu.
5. Huấn luyện với AdamW, gradient clipping và early stopping.
6. Chọn checkpoint bằng dev entity-level F1.
7. Đánh giá test set và phân tích lỗi BIO.

### Bài tập mở rộng

1. So sánh `num_heads ∈ {2, 4, 8}` khi giữ nguyên `d_model`.
2. Bỏ positional encoding và đo mức giảm F1.
3. Thử class weight để giảm ảnh hưởng của nhãn `O`.
4. Thêm tầng CRF lên trên Transformer.
5. So sánh mô hình từ đầu với một Transformer tiền huấn luyện cho tiếng Việt.


In [ ]:
def extract_entities(
    tokens: list[str],
    tags: list[str],
) -> list[dict[str, Any]]:
    entities = []

    for entity_type, start, end in get_entities(tags):
        entities.append(
            {
                "type": entity_type,
                "start": start,
                "end": end,
                "text": " ".join(tokens[start : end + 1]),
            }
        )

    return entities


@torch.no_grad()
def predict_tokens(
    model: nn.Module,
    tokens: list[str],
    token_to_id: dict[str, int],
    id_to_tag: list[str],
    device: torch.device,
    max_length: int,
) -> dict[str, Any]:
    if not tokens:
        raise ValueError(
            "Danh sách token không được rỗng."
        )

    truncated_tokens = tokens[:max_length]
    input_ids = torch.tensor(
        [
            token_to_id.get(
                normalize_token(token),
                UNK_ID,
            )
            for token in truncated_tokens
        ],
        dtype=torch.long,
        device=device,
    ).unsqueeze(0)

    attention_mask = input_ids.ne(PAD_ID)

    model.eval()
    logits = model(
        input_ids,
        attention_mask,
    )
    predicted_ids = logits.argmax(dim=-1)[0].tolist()
    predicted_tags = [
        id_to_tag[tag_id]
        for tag_id in predicted_ids
    ]

    return {
        "tokens": truncated_tokens,
        "tags": predicted_tags,
        "entities": extract_entities(
            truncated_tokens,
            predicted_tags,
        ),
        "was_truncated": len(tokens) > max_length,
    }


example_tokens = [
    "Bệnh",
    "nhân",
    "được",
    "điều",
    "trị",
    "tại",
    "Bệnh",
    "viện",
    "Chợ",
    "Rẫy",
    ".",
]

prediction = predict_tokens(
    model=best_model,
    tokens=example_tokens,
    token_to_id=best_checkpoint["token_to_id"],
    id_to_tag=best_checkpoint["id_to_tag"],
    device=DEVICE,
    max_length=best_checkpoint["max_length"],
)

print("Dự đoán theo token:")
for token, tag in zip(
    prediction["tokens"],
    prediction["tags"],
):
    print(f"{token:<20} {tag}")

print("\nCác thực thể:")
for entity in prediction["entities"]:
    print(entity)
